In [2]:
import os
import yfinance as yf
from sec_edgar_downloader import Downloader
import requests
import pandas as pd
from newsapi import NewsApiClient

# 환경변수에서 API 키 읽기
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
NEWSAPI_API_KEY        = os.getenv("NEWSAPI_API_KEY")

tickers = ["AAPL","AMZN","NVDA","MSFT","GOOGL"]

In [3]:
for t in tickers:
    ticker = yf.Ticker(t)
    hist = ticker.history(period="1y")  # 과거 1년 일별 시세
    print(f"{t} - 최근 종가:\n", hist["Close"].tail(3), "\n")

AAPL - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    213.880005
2025-07-28 00:00:00-04:00    214.050003
2025-07-29 00:00:00-04:00    212.580002
Name: Close, dtype: float64 

AMZN - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    231.440002
2025-07-28 00:00:00-04:00    232.789993
2025-07-29 00:00:00-04:00    230.980103
Name: Close, dtype: float64 

NVDA - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    173.500000
2025-07-28 00:00:00-04:00    176.750000
2025-07-29 00:00:00-04:00    177.235001
Name: Close, dtype: float64 

MSFT - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    513.710022
2025-07-28 00:00:00-04:00    512.500000
2025-07-29 00:00:00-04:00    513.109985
Name: Close, dtype: float64 

GOOGL - 최근 종가:
 Date
2025-07-25 00:00:00-04:00    193.179993
2025-07-28 00:00:00-04:00    192.580002
2025-07-29 00:00:00-04:00    193.399597
Name: Close, dtype: float64 



In [ ]:
from sec_edgar_downloader import Downloader

dl = Downloader(
    "GEN",                
    "james4327@gmail.com",            
    download_folder="sec"     
)

for t in tickers:
    dl.get("10-K", t, limit=1)

print("10-K 다운로드 완료 → notebook/ 폴더 확인")

10-K 다운로드 완료 → notebook/ 폴더 확인


In [14]:
import os
import pandas as pd

# 간단 청킹 함수 정의
def chunk_text(text, size=1000, overlap=200):
    chunks = []
    start = 0
    length = len(text)
    while start < length:
        end = min(start + size, length)
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

# 설정
tickers = ["AAPL","AMZN","NVDA","MSFT","GOOGL"]
records = []
base_path = os.path.join("sec_filings_nb", "sec-edgar-filings")

# full-submission.txt 읽어서 청킹
for ticker in tickers:
    tenk_dir = os.path.join(base_path, ticker, "10-K")
    if not os.path.isdir(tenk_dir):
        continue
    # 각 종목의 10-K 폴더 내 유일한 서브디렉터리 찾기
    subs = [d for d in os.listdir(tenk_dir)
            if os.path.isdir(os.path.join(tenk_dir, d))]
    for sub in subs:
        txt_file = os.path.join(tenk_dir, sub, "full-submission.txt")
        if not os.path.isfile(txt_file):
            continue
        with open(txt_file, 'r', encoding='utf-8') as f:
            text = f.read()
        # 청킹
        chunks = chunk_text(text)
        for idx, chunk in enumerate(chunks):
            records.append({
                "ticker": ticker,
                "chunk_id": idx,
                "content": chunk
            })

# DataFrame 생성 및 샘플 출력
df_chunks = pd.DataFrame(records)
print("총 청크 수:", len(df_chunks))

총 청크 수: 99084


In [15]:
df_chunks

,ticker,chunk_id,content
0,AAPL,0,<SEC-DOCUMENT>0000320193-24-000123.txt : 20241...
1,AAPL,1,\t\tCUPERTINO\n\t\tSTATE:\t\t\tCA\n\t\tZIP:\t\...
2,AAPL,2,"so4217"" xmlns:country=""http://xbrl.sec.gov/cou..."
3,AAPL,3,"v style=""display:none""><ix:header><ix:hidden><..."
4,AAPL,4,anceObligationExpectedTimingOfSatisfactionPeri...
...,...,...,...
99079,GOOGL,18463,"v&gt;&lt;div style=""margin-top:3pt;text-align:..."
99080,GOOGL,18464,"mily:'Arial',sans-serif;font-size:9pt;font-wei..."
99081,GOOGL,18465,"6"" id=""f-1712"" unitRef=""usd"">2301000000</us-ga..."
99082,GOOGL,18466,ons>\n <us-gaap:ValuationAllowancesAndReser...
